In [1]:
import sys
import pandas as pd

print(sys.version)
print(pd.__version__)

3.14.6 (main, Aug  4 2026, 19:23:56) [Clang 22.1.3 ]
3.0.5


## 1. Load the loans and look for gaps

In [2]:
loans_raw = pd.read_csv(
    'data/library_loans_clean.csv',
    parse_dates=['loan_date', 'return_date']
)
loans = loans_raw.copy()

print(loans.shape)
print(loans.dtypes)

(600, 8)
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object


In [3]:
loans.isna().sum()

loan_id         0
member_id       0
title           0
genre           0
loan_date       3
return_date    36
late_fee        0
returned        0
dtype: int64

In [4]:
(loans.isna().mean() * 100).round(1)

loan_id        0.0
member_id      0.0
title          0.0
genre          0.0
loan_date      0.5
return_date    6.0
late_fee       0.0
returned       0.0
dtype: float64

It has 600 rows and 8 columns. loan_date has 3 missing values (0.5%) and return_date has 36 (6.0%).

## 2. Two kinds of missing

In [5]:
loans[loans['loan_date'].isna()]

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
462,L-30463,M0007,Blue Moon Protocol,Sci-fi,NaT,2026-05-24,0.0,True
467,L-30468,M0018,Blue Moon Protocol,Sci-fi,NaT,2026-06-08,0.0,True
571,L-30572,M0090,Mimmi and the Moon,Children,NaT,2026-06-26,0.0,True


In [6]:
missing_return_date = loans['return_date'].isna()

print((missing_return_date & (loans['returned'] == True)).sum())
print((missing_return_date & (loans['returned'] == False)).sum())

6
30


In [7]:
loans['has_return_date'] = loans['return_date'].notna()

loans['has_return_date'].value_counts()

has_return_date
True     564
False     36
Name: count, dtype: int64

The 30 loans that are not back have no return date for a reason. The 6 returned loans with no return date suggest missing records by accident. has_return_date is True for 564 loans and False for 36.

## 3. Look at the members

In [8]:
members_raw = pd.read_csv('data/library_members.csv')
members = members_raw.copy()

print(members.shape)

(148, 3)


In [9]:
members.info()

<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB


In [10]:
members['birth_year'].describe().round(2)

count     130.00
mean     1983.38
std        25.16
min      1900.00
25%      1978.25
50%      1988.00
75%      1996.75
max      2030.00
Name: birth_year, dtype: float64

In [11]:
print((members['birth_year'] == 1900).sum())
print((members['birth_year'] == 2030).sum())

8
3


In [12]:
print(members.duplicated().sum())
print(members.duplicated(subset='member_id').sum())

4
4


There are 148 rows and 3 columns. Problems: 12 missing branches, 18 missing birth years, 8 placeholder birth years of 1900, 3 impossible birth years of 2030, and 4 duplicate rows with duplicate member IDs.

## 4. Placeholders and impossible values

In [13]:
members['birth_year'] = members['birth_year'].where(
    members['birth_year'].between(1920, 2015)
)

members['birth_year'].isna().sum()

np.int64(29)

In [14]:
members['birth_year'].describe().round(2)

count     119.00
mean     1987.82
std        12.06
min      1957.00
25%      1979.00
50%      1988.00
75%      1996.50
max      2010.00
Name: birth_year, dtype: float64

There are now 29 missing birth years. The remaining 119 values have mean 1987.82, minimum 1957, and maximum 2010. Before cleaning, the mean was about 1983.4, with minimum 1900 and maximum 2030.

## 5. Remove, fill, or leave

(a) Compare dropping rows without changing members

In [15]:
print(members.dropna().shape)
print(members.dropna(subset=['branch']).shape)
print(members.dropna(subset=['birth_year']).shape)

(107, 3)
(136, 3)
(119, 3)


The shapes are (107, 3), (136, 3), and (119, 3).

(b) Fill missing branches

In [16]:
members['branch'] = members['branch'].fillna('Unknown')

members['branch'].value_counts()

branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64

(c) Try median filling separately

In [17]:
median_birth_year = members['birth_year'].median()
birth_year_filled = members['birth_year'].fillna(median_birth_year)

print('Median:', median_birth_year)
print('Mean before:', members['birth_year'].mean().round(2))
print('Mean after:', birth_year_filled.mean().round(2))
print('Standard deviation before:', members['birth_year'].std().round(2))
print('Standard deviation after:', birth_year_filled.std().round(2))

Median: 1988.0
Mean before: 1987.82
Mean after: 1987.85
Standard deviation before: 12.06
Standard deviation after: 10.81


The median is 1988. The mean changes from 1987.82 to 1987.85, and the standard deviation falls from 12.06 to 10.81. Filling gaps with the same middle value makes the years look less spread out. I leave birth_year missing because the real years are unknown.

## 6. Merge, and check

In [18]:
print(loans.shape)
print(members.shape)

(600, 9)
(148, 3)


In [19]:
merged_unchecked = loans.merge(members, on='member_id', how='left')

print(merged_unchecked.shape)
print(loans['late_fee'].sum())
print(merged_unchecked['late_fee'].sum())

(616, 11)
7487.5
7725.0


The merge has 616 rows rather than 600 because duplicate member IDs repeat matching loans. Fees increase from 7,487.5 to 7,725.0 kr, an incorrect extra 237.5 kr.

Intentional error: validation detects duplicate keys in members.

In [20]:
loans.merge(
    members, on='member_id', how='left', validate='many_to_one'
)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 member_id
    M0015
    M0033
    M0065
    M0071 ...

## 7. Fix the key, and find who is missing

In [21]:
members = members.drop_duplicates().reset_index(drop=True)

print(members.shape)

(144, 3)


In [22]:
merged = loans.merge(
    members,
    on='member_id',
    how='left',
    validate='many_to_one',
    indicator=True
)

print(merged.shape)
print(merged['late_fee'].sum())

(600, 12)
7487.5


In [23]:
merged['_merge'].value_counts()

_merge
both          569
left_only      31
right_only      0
Name: count, dtype: int64

In [24]:
missing_members = merged[merged['_merge'] == 'left_only']

print(len(missing_members))
print(missing_members['member_id'].nunique())
print(missing_members['late_fee'].sum())

31
6
347.5


In [25]:
inner = loans.merge(
    members, on='member_id', how='inner', validate='many_to_one'
)

print(inner.shape)
print(inner['late_fee'].sum())

(569, 11)
7140.0


The deduplicated member table has 144 rows. The left merge has 600 rows and 12 columns, including the merge indicator, and preserves 7,487.5 kr in fees. There are 31 unmatched loans from 6 members, with 347.5 kr in fees. The inner merge excludes those loans, giving (569, 11) and 7,140.0 kr.

## 8. Use the merged table

In [26]:
merged['branch'] = merged['branch'].fillna('No member record')

merged.groupby('branch')['loan_id'].count().sort_values(ascending=False)

branch
Central             144
South               130
North               126
East                 72
Harbour              51
Unknown              46
No member record     31
Name: loan_id, dtype: int64

Central has the most loans (144), followed by South (130), North (126), East (72), Harbour (51), Unknown (46), and No member record (31). Unknown means the member record exists but its branch was missing; No member record means the loan did not match any member record.